In [ ]:
%cd ../
%ls

In [ ]:
from wag_toolkit.locations import Locations
import pandas as pd
import json
from collections import Counter
from dotenv import load_dotenv
import awswrangler as wr

# Load environment variables from .env file
load_dotenv()

In [ ]:
grants_ref =  wr.s3.read_excel('s3://datalabs-data/funding_impact_measures/climate_and_health/Manual Tagging of Grants with C&H goals.xlsx', sheet_name='Total C&H Awards_SM tagging')
grants_ref = grants_ref[grants_ref['C&H/C&H in partnership/wider portfolio (new)'].isin(['C&H', 'C&H in partnership'])]
grants_list = list(set(grants_ref['Grant Reference'].to_list()))

In [ ]:
# Only Wellcome
candh_intersection =  wr.s3.read_excel('s3://datalabs-data/funding_impact_measures/climate_and_health/C&HPublicationExtraction.xlsx', sheet_name='Combined Awards')


# All C&H
# candh_intersection =  wr.s3.read_parquet('s3://datalabs-data/funding_impact_measures/climate_and_health/climate_health_intersection_just_ids.parquet')

In [ ]:
# Select relevant Grants for C&H
candh_intersection = candh_intersection[candh_intersection['Wellcome Grant Reference'].isin(grants_list)]

pub_ids = list(set(candh_intersection['Publication ID'].tolist()))

In [ ]:
dummy_query = """MATCH (r:Researcher)-[a:AUTHORED]->(p:Publication) RETURN * LIMIT 1"""
loc = Locations(dummy_query)

query = """MATCH (r:Researcher)-[a:AUTHORED]->(p:Publication)
            WHERE p.dimensions_publication_id IN {}
            RETURN p.dimensions_publication_id AS dimensions_publication_id, p.year AS year,
                a.institutions AS grid_id"""
loc.lookup_query(query=query, lookup=pub_ids)

In [ ]:
data =pd.DataFrame(loc.data)
data = data.explode("grid_id").dropna()
data.head()

In [ ]:
loc._clean_grid_ids()
loc.extract_edges()
loc.extract_locations("country")

loc.convert_edges()
loc.calculate_adjacency_matrices()

In [ ]:
import numpy as np
loc.adjacency_matrices = {np.int64(key): value for key, value in loc.adjacency_matrices.items()}

In [ ]:
countries_class = pd.read_excel('geographies/files/CLASS.xlsx', sheet_name='List of economies')
lmic_list = countries_class[countries_class['Income group'].isin(['Low income', 'Lower middle income', 'Upper middle income'])]
lmic_list.head()

In [ ]:
lmic_list['Economy'] = lmic_list['Economy'].replace("Côte d’Ivoire", "Ivory Coast")
lmic_list['Economy'] = lmic_list['Economy'].replace("Gambia, The", "Gambia")
lmic_list['Economy'] = lmic_list['Economy'].replace("Iran, Islamic Rep.", "Iran")

In [ ]:
# Only account for LMIC
only_lmic = True

if only_lmic:
    for year in loc.adjacency_matrices:
        for country in loc.adjacency_matrices[year]['All'].index:
            if country == 'All' or country == 'total':
                continue
            for country2 in loc.adjacency_matrices[year]['All'][country].index:
                if country in list(lmic_list['Economy']) or country2 in list(lmic_list['Economy']):
                    continue
                else:
                    loc.adjacency_matrices[year]['All'][country][country2]=0

In [ ]:
loc.load_visjs_nodes_and_edges(node_scaling=0.0015, edge_scaling=0.5, directed=False, threshold=0, font = {"size": 20, "face": "Helvetica Neue"}, node_count='total')

In [ ]:
dirname = 'CH_LMIC_Countries2'
loc.to_visjs(vis_name="locations", directed=False, template='geographies/locations.html', dirname=dirname)
loc._to_json("nodes_all.json", loc.vis_nodes)
loc._to_json("edges_all.json", loc.vis_edges)

In [ ]:
import os
import shutil

with(open(f'{dirname}/edges.json','r')) as f:
    edges = json.load(f)

os.rename(f'{dirname}/nodes.json', f'{dirname}/nodes_all.json')
shutil.copyfile('geographies/positions.json', f'{dirname}/positions.json')

def edges_to_dict(edges):
    edge_dict = {}
    for e in edges:
        year = e['year']
        _ = e.pop('year')
        if edge_dict.get(year):
            edge_dict[year].append(e)
        else:
            edge_dict[year] = [e]
    return edge_dict


with(open(f'{dirname}/dict_edges_all.json','w')) as f:
        json.dump(edges_to_dict(edges), f)